# Task 2: Sales Data Analysis — Inspection, KPIs, and Validation

## Objective
Inspect the cleaned e-commerce dataset, calculate the required sales KPIs, validate calculations independently, and prepare monthly/category trend tables for interpretation and Power BI.

**Important date-quality decision:** six records have an order date of 15 January 2035. The notebook flags dates after the fixed analysis cutoff (13 October 2026) and excludes them from time-based trends only. It does not invent replacement dates or overwrite the original dataset. Non-date KPIs continue to include those rows unless the business/data owner confirms the records themselves are invalid.

In [1]:
# Mount Google Drive only when running in Google Colab; local Jupyter can use the fallback file path.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Google Colab not detected; using the local dataset path if available.")

Mounted at /content/drive


## 1. Import libraries and load the cleaned dataset

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Update this path if your Excel file is stored elsewhere.
FILE_PATH = Path("/content/drive/MyDrive/data-science-week1-assignment/data/cleaned/ecommerce_cleaned_dataset.xlsx")
if not FILE_PATH.exists():
    FILE_PATH = Path("ecommerce_cleaned_dataset.xlsx")

if not FILE_PATH.exists():
    raise FileNotFoundError(
        "Could not find ecommerce_cleaned_dataset.xlsx. "
        "Update FILE_PATH to the actual location of the file."
    )

xls = pd.ExcelFile(FILE_PATH)
SHEET_NAME = "Cleaned_Data" if "Cleaned_Data" in xls.sheet_names else xls.sheet_names[0]
df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

print(f"Loaded: {FILE_PATH}")
print(f"Worksheet: {SHEET_NAME}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Loaded: /content/drive/MyDrive/data-science-week1-assignment/data/cleaned/ecommerce_cleaned_dataset.xlsx
Worksheet: Cleaned_Data
Rows: 10,009
Columns: 16


## 2. Review first rows, column names, dimensions, and data types

In [3]:
display(df.head())
print("Dataset shape (rows, columns):", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types and non-null counts:")
df.info()

,order_id,order_date,customer_id,customer_name,product_id,product_name,category,subcategory,quantity,unit_price,total_amount,discount,payment_mode,delivery_status,city,region
0,CA-2017-151428,2017-09-21,RH-19495,Rick Hansen,OFF-BI-10000546,Avery Durable Binders,Office Supplies,Binders,7.0,2.88,20.16,0.0,Debit Card,Delivered,Rochester,Central
1,CA-2017-137470,2017-09-17,TP-21415,Tom Prescott,OFF-PA-10002001,Xerox 1984,Office Supplies,Paper,2.0,6.48,12.96,0.0,Credit Card,Delivered,Seattle,West
2,CA-2017-138779,2017-01-14,RB-19570,Rob Beeghly,OFF-EN-10002504,"Tyvek Top-Opening Peel & Seel Envelopes, Plain...",Office Supplies,Envelopes,1.0,21.74,21.74,0.2,Debit Card,Delivered,Jacksonville,South
3,CA-2015-149678,2015-04-13,AW-10840,Anthony Witt,OFF-SU-10004498,Martin-Yale Premier Letter Opener,Office Supplies,Supplies,1.0,12.88,12.88,0.0,Credit Card,Returned,Farmington,West
4,CA-2016-146682,2016-10-29,KW-16435,Katrina Willman,FUR-FU-10002671,Electrix 20W Halogen Replacement Bulb for Zoom...,Furniture,Furnishings,5.0,13.40,67.00,0.0,Debit Card,Processing,Lansing,Central


Dataset shape (rows, columns): (10009, 16)
\nColumn names:
['order_id', 'order_date', 'customer_id', 'customer_name', 'product_id', 'product_name', 'category', 'subcategory', 'quantity', 'unit_price', 'total_amount', 'discount', 'payment_mode', 'delivery_status', 'city', 'region']
\nData types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10009 entries, 0 to 10008
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         10009 non-null  object        
 1   order_date       9942 non-null   datetime64[ns]
 2   customer_id      9950 non-null   object        
 3   customer_name    9890 non-null   object        
 4   product_id       10009 non-null  object        
 5   product_name     9930 non-null   object        
 6   category         9889 non-null   object        
 7   subcategory      10009 non-null  object        
 8   quantity         9930 non-null   float64

## 3. Review missing values

In [4]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values(
    "missing_count", ascending=False
)
display(missing_summary)
print("Total missing cells:", int(df.isna().sum().sum()))

,missing_count,missing_percentage
payment_mode,179,1.79
delivery_status,179,1.79
city,120,1.20
unit_price,120,1.20
category,120,1.20
customer_name,119,1.19
quantity,79,0.79
product_name,79,0.79
order_date,67,0.67
customer_id,59,0.59


Total missing cells: 1121


## 4. Review duplicates and repeated order IDs

In [5]:
print("Fully duplicated rows:", int(df.duplicated().sum()))

if "order_id" in df.columns:
    print("Total rows:", len(df))
    print("Unique non-null order IDs:", int(df["order_id"].nunique(dropna=True)))
    print("Rows after the first occurrence of an order ID:", int(df["order_id"].duplicated().sum()))
    print("Missing order IDs:", int(df["order_id"].isna().sum()))

# Repeated order IDs may represent multiple product lines in one order.
# Do not remove them without confirming they are accidental duplicates.

Fully duplicated rows: 0
Total rows: 10009
Unique non-null order IDs: 5009
Rows after the first occurrence of an order ID: 5000
Missing order IDs: 0


## 5. Review date range and unusual dates

This dataset contains six rows dated **15 January 2035**, which is in the future relative to the analysis cutoff of **13 October 2026**. The notebook does not guess replacement dates. It preserves the original `df` and creates an analysis-only copy in which future dates are set to `NaT` for date-based trend analysis. These rows remain included in non-date KPIs because their revenue, quantities, and product details can still be used; confirm the original dates with the data owner before making any permanent correction.

In [6]:
if "order_date" in df.columns:
    parsed_dates = pd.to_datetime(df["order_date"], errors="coerce")
    print("Missing or unparseable dates:", int(parsed_dates.isna().sum()))
    print("Earliest parsed date:", parsed_dates.min())
    print("Latest parsed date:", parsed_dates.max())

    # A fixed cutoff makes this analysis reproducible. Change it if your project uses
    # a different approved reporting date.
    AS_OF_DATE = pd.Timestamp("2026-10-13")
    future_date_mask = parsed_dates > AS_OF_DATE
    future_date_rows = df.loc[future_date_mask].copy()

    print(f"Analysis cutoff date: {AS_OF_DATE.date()}")
    print("Rows dated after the cutoff:", int(future_date_mask.sum()))
    display(future_date_rows[[
        c for c in ["order_id", "order_date", "customer_id", "product_name",
                    "quantity", "total_amount", "delivery_status"] if c in df.columns
    ]])

    # Do not overwrite df or the Excel source. Mark suspicious dates as missing
    # only in the analysis copy used for time-based reporting.
    analysis_df = df.copy()
    analysis_df["order_date"] = parsed_dates
    analysis_df.loc[future_date_mask, "order_date"] = pd.NaT
    print("Rows retained for non-date KPIs:", len(analysis_df))
    print("Rows eligible for date-based trends:", int(analysis_df["order_date"].notna().sum()))
else:
    print("No order_date column found.")


Missing or unparseable dates: 67
Earliest parsed date: 2014-01-03 00:00:00
Latest parsed date: 2035-01-15 00:00:00
Analysis cutoff date: 2026-10-13
Rows dated after the cutoff: 6


,order_id,order_date,customer_id,product_name,quantity,total_amount,delivery_status
25,CA-2017-103009,2035-01-15,PJ-18835,Xerox 1963,5.0,26.40,Delivered
2150,CA-2017-128755,2035-01-15,MK-18160,"Black Print Carbonless Snap-Off Rapid Letter, ...",1.0,9.11,Delivered
2275,CA-2017-127264,2035-01-15,SA-20830,Prang Colored Pencils,3.0,7.06,Delivered
2893,CA-2014-129364,2035-01-15,TB-21250,Xerox 232,2.0,10.37,Processing
4695,CA-2016-128531,2035-01-15,NS-18505,Logitech G105 Gaming Keyboard,2.0,94.99,Delivered
8804,CA-2016-150350,2035-01-15,MS-17770,"Tyvek Top-Opening Peel & Seel Envelopes, Plain...",5.0,135.90,Cancelled


Rows retained for non-date KPIs: 10009
Rows eligible for date-based trends: 9936


## 6. Review numeric values

In [7]:
numeric_candidates = [
    col for col in ["quantity", "unit_price", "total_amount", "discount"]
    if col in df.columns
]
display(df[numeric_candidates].describe().T)

checks = []
for col in ["quantity", "unit_price", "total_amount", "discount"]:
    if col in df.columns:
        values = pd.to_numeric(df[col], errors="coerce")
        item = {
            "column": col,
            "negative_values": int((values < 0).sum()),
            "zero_values": int((values == 0).sum()),
            "missing_or_non_numeric": int(values.isna().sum())
        }
        if col == "discount":
            item["discount_above_1"] = int((values > 1).sum())
        checks.append(item)
display(pd.DataFrame(checks))

,count,mean,std,min,25%,50%,75%,max
quantity,9930.0,3.777241,2.243538,-5.00,2.00,3.00,5.00,14.00
unit_price,9889.0,60.839665,143.179398,-12.50,5.42,16.24,63.69,3773.08
total_amount,10009.0,229.859398,623.049080,0.44,17.28,54.48,209.88,22638.48
discount,10009.0,0.156109,0.206367,0.00,0.00,0.20,0.20,0.80


,column,negative_values,zero_values,missing_or_non_numeric,discount_above_1
0,quantity,14,10,79,NaN
1,unit_price,12,8,120,NaN
2,total_amount,0,0,0,NaN
3,discount,0,4807,0,0.0


## 7. Review delivery-status categories

In [8]:
if "delivery_status" in df.columns:
    display(
        df["delivery_status"]
        .value_counts(dropna=False)
        .rename_axis("delivery_status")
        .to_frame("count")
    )
else:
    print("No delivery_status column found.")

,count
delivery_status,
Delivered,6307
Shipped,1482
Processing,833
Cancelled,688
Returned,488
NaN,179
In Transit,22
Return,10


## 8. Prepare trend-analysis data

Use `analysis_df` for monthly and category-over-time trends. Rows with dates after the analysis cutoff are excluded from time-based groupings because their dates are unverified. The original `df` remains unchanged, and those rows remain in the baseline non-date KPI calculations below.

In [9]:
# Monthly trend: sum revenue and units, count distinct orders.
# Future-dated/unparseable rows are excluded from date-based trends only.
trend_source = analysis_df.loc[analysis_df["order_date"].notna()].copy()
trend_source["_revenue"] = pd.to_numeric(trend_source["total_amount"], errors="coerce")
trend_source["_quantity"] = pd.to_numeric(trend_source["quantity"], errors="coerce")
trend_source["month"] = trend_source["order_date"].dt.to_period("M").dt.to_timestamp()

monthly_trend = (
    trend_source.groupby("month", as_index=False)
    .agg(
        revenue=("_revenue", "sum"),
        units_sold=("_quantity", "sum"),
        distinct_orders=("order_id", "nunique"),
    )
    .sort_values("month")
)
monthly_trend["revenue_change_pct"] = monthly_trend["revenue"].pct_change().mul(100)
display(monthly_trend.tail(12))

category_monthly_trend = (
    trend_source.loc[trend_source["category"].notna()]
    .groupby(["month", "category"], as_index=False)
    .agg(revenue=("_revenue", "sum"), units_sold=("_quantity", "sum"))
    .sort_values(["month", "category"])
)
print("Category-by-month trend rows:", len(category_monthly_trend))
display(category_monthly_trend.tail(10))


,month,revenue,units_sold,distinct_orders,revenue_change_pct
36,2017-01-01,43974.61,587.0,69,-54.861155
37,2017-02-01,19968.99,361.0,52,-54.589728
38,2017-03-01,44829.71,871.0,118,124.496632
39,2017-04-01,36173.52,727.0,115,-19.309048
40,2017-05-01,43926.94,877.0,118,21.433966
41,2017-06-01,52730.23,918.0,133,20.040754
42,2017-07-01,44607.83,812.0,111,-15.403688
43,2017-08-01,63523.53,872.0,110,42.404439
44,2017-09-01,88230.61,1630.0,225,38.894375
45,2017-10-01,76068.00,1104.0,147,-13.785023


Category-by-month trend rows: 144


,month,category,revenue,units_sold
134,2017-09-01,Technology,26695.66,302.0
135,2017-10-01,Furniture,20077.59,242.0
136,2017-10-01,Office Supplies,23043.50,632.0
137,2017-10-01,Technology,32346.86,225.0
138,2017-11-01,Furniture,37005.75,353.0
139,2017-11-01,Office Supplies,29876.25,1027.0
140,2017-11-01,Technology,49776.07,376.0
141,2017-12-01,Furniture,30887.70,375.0
142,2017-12-01,Office Supplies,30444.01,1027.0
143,2017-12-01,Technology,21916.75,267.0


## 9. Task 2 KPI calculations

### KPI definitions and assumptions
- **Total revenue:** sum of `total_amount` across all rows in the cleaned dataset. This is the baseline reported value; it includes every delivery status because the dataset does not specify an approved revenue-recognition rule.
- **Total orders:** distinct non-null `order_id` values. Repeated order IDs are treated as multiple line items belonging to one order, not automatically as duplicate records.
- **Average order value (AOV):** baseline total revenue divided by distinct orders.
- **Total units sold:** sum of non-missing `quantity` values.
- **Top-selling category:** category with the highest summed quantity. Rows with a missing category are excluded from the category ranking and are reported separately in the QA checks.
- **Top revenue-generating product:** product with the highest summed `total_amount`, grouped by `product_id` and `product_name`. If a product name is missing, the product ID is retained so its revenue is not silently dropped.

**Interpretation caution:** `delivery_status` includes `Cancelled`, `Returned`, `Return`, and missing values. Since the dataset does not define whether these amounts should be included in revenue, the notebook also reports a sensitivity view excluding those statuses. Do not substitute that alternative for the baseline without confirming the business rule.

In [10]:
# Validate that the fields required for the KPIs are available.
required_columns = {
    "order_id", "total_amount", "quantity", "category",
    "product_id", "product_name", "delivery_status"
}
missing_required = required_columns.difference(df.columns)
if missing_required:
    raise KeyError(f"Missing required KPI columns: {sorted(missing_required)}")

# Normalize key numeric fields for calculations without changing the source DataFrame.
revenue_values = pd.to_numeric(df["total_amount"], errors="coerce")
quantity_values = pd.to_numeric(df["quantity"], errors="coerce")

total_revenue = float(revenue_values.sum(skipna=True))
total_orders = int(df["order_id"].nunique(dropna=True))
average_order_value = total_revenue / total_orders if total_orders else np.nan
total_units_sold = float(quantity_values.sum(skipna=True))

# Category rank is based on units, not revenue.
category_kpis = (
    df.loc[df["category"].notna()]
      .assign(_quantity=quantity_values, _revenue=revenue_values)
      .groupby("category", as_index=False)
      .agg(units_sold=("_quantity", "sum"),
           revenue=("_revenue", "sum"),
           line_items=("order_id", "size"))
      .sort_values(["units_sold", "revenue"], ascending=[False, False])
      .reset_index(drop=True)
)

# Group by product ID as well as name to avoid combining distinct products that share a name.
product_source = df.copy()
product_source["_revenue"] = revenue_values
product_source["_quantity"] = quantity_values
product_source["_product_name_display"] = product_source["product_name"].fillna(
    "Unknown product name"
)
product_kpis = (
    product_source.groupby(
        ["product_id", "_product_name_display"], dropna=False, as_index=False
    )
    .agg(revenue=("_revenue", "sum"),
         units_sold=("_quantity", "sum"),
         line_items=("order_id", "size"))
    .rename(columns={"_product_name_display": "product_name"})
    .sort_values(["revenue", "units_sold"], ascending=[False, False])
    .reset_index(drop=True)
)

kpi_summary = pd.DataFrame([
    {"KPI": "Total Revenue (all statuses)", "Value": total_revenue,
     "Definition": "Sum of total_amount across all rows"},
    {"KPI": "Total Orders", "Value": total_orders,
     "Definition": "Distinct non-null order_id values"},
    {"KPI": "Average Order Value (AOV)", "Value": average_order_value,
     "Definition": "Total revenue divided by distinct orders"},
    {"KPI": "Total Units Sold", "Value": total_units_sold,
     "Definition": "Sum of non-missing quantity values"},
    {"KPI": "Top-Selling Category (by units)", "Value": category_kpis.iloc[0]["category"] if not category_kpis.empty else None,
     "Definition": "Category with the highest summed quantity"},
    {"KPI": "Top Revenue-Generating Product", "Value": product_kpis.iloc[0]["product_name"] if not product_kpis.empty else None,
     "Definition": "Product with the highest summed total_amount"},
])

display(kpi_summary)
print("\nTop categories by units sold:")
display(category_kpis.head(5))
print("\nTop products by revenue:")
display(product_kpis.head(5))


,KPI,Value,Definition
0,Total Revenue (all statuses),2300662.71,Sum of total_amount across all rows
1,Total Orders,5009,Distinct non-null order_id values
2,Average Order Value (AOV),459.305792,Total revenue divided by distinct orders
3,Total Units Sold,37508.0,Sum of non-missing quantity values
4,Top-Selling Category (by units),Office Supplies,Category with the highest summed quantity
5,Top Revenue-Generating Product,Canon imageCLASS 2200 Advanced Copier,Product with the highest summed total_amount



Top categories by units sold:


,category,units_sold,revenue,line_items
0,Office Supplies,22297.0,711444.99,5938
1,Furniture,7918.0,739425.92,2112
2,Technology,6845.0,832211.05,1839



Top products by revenue:


,product_id,product_name,revenue,units_sold,line_items
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61599.83,20.0,5
1,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,31.0,10
2,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,22638.48,6.0,1
3,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,21870.57,39.0,8
4,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,19823.47,37.0,11


## 10. KPI validation and revenue-status sensitivity

The checks below independently recompute the headline measures and verify that the revenue grouped by delivery status reconciles to the dataset total. Assertions stop execution if the core calculations disagree. The status sensitivity table is a decision aid, not a replacement business definition.

In [11]:
# Independent cross-checks for the headline KPI calculations.
checks = []

def record_check(name, passed, detail):
    checks.append({"Check": name, "Passed": bool(passed), "Details": str(detail)})

independent_revenue = float(np.nansum(pd.to_numeric(df["total_amount"], errors="coerce").to_numpy()))
independent_orders = int(len(pd.unique(df.loc[df["order_id"].notna(), "order_id"])))
independent_units = float(np.nansum(pd.to_numeric(df["quantity"], errors="coerce").to_numpy()))
independent_aov = independent_revenue / independent_orders if independent_orders else np.nan

record_check("Revenue matches independent sum",
             np.isclose(total_revenue, independent_revenue, rtol=0, atol=0.01),
             f"calculated={total_revenue:.2f}; independent={independent_revenue:.2f}")
record_check("Order count matches distinct-ID count",
             total_orders == independent_orders,
             f"calculated={total_orders}; independent={independent_orders}")
record_check("Units match independent sum",
             np.isclose(total_units_sold, independent_units, rtol=0, atol=0.01),
             f"calculated={total_units_sold:.2f}; independent={independent_units:.2f}")
record_check("AOV formula reconciles",
             np.isclose(average_order_value, independent_aov, rtol=0, atol=0.01),
             f"calculated={average_order_value:.2f}; independent={independent_aov:.2f}")

# Reconciliation by status, retaining missing statuses as their own group.
status_reconciliation = (
    df.assign(_revenue=revenue_values)
      .groupby("delivery_status", dropna=False, as_index=False)
      .agg(line_items=("order_id", "size"), revenue=("_revenue", "sum"))
)
status_revenue_sum = float(status_reconciliation["revenue"].sum())
record_check("Revenue by status reconciles to total",
             np.isclose(total_revenue, status_revenue_sum, rtol=0, atol=0.01),
             f"dataset total={total_revenue:.2f}; status sum={status_revenue_sum:.2f}")

# Basic data-quality context relevant to interpretation.
record_check("Order IDs are non-null",
             bool(df["order_id"].notna().all()),
             f"missing order IDs={int(df['order_id'].isna().sum())}")
record_check("Revenue values are non-negative",
             bool((revenue_values.dropna() >= 0).all()),
             f"negative revenue rows={int((revenue_values < 0).sum())}")

# Aggregation-level checks for category and product rankings.
category_units_total = float(category_kpis["units_sold"].sum())
category_revenue_total = float(category_kpis["revenue"].sum())
record_check(
    "Category aggregation reconciles to rows with a known category",
    np.isclose(category_units_total,
               float(pd.to_numeric(df.loc[df["category"].notna(), "quantity"], errors="coerce").sum()),
               rtol=0, atol=0.01),
    f"category units total={category_units_total:.2f}"
)
record_check(
    "Product aggregation reconciles to total revenue",
    np.isclose(float(product_kpis["revenue"].sum()), total_revenue, rtol=0, atol=0.01),
    f"product revenue total={float(product_kpis['revenue'].sum()):.2f}; total revenue={total_revenue:.2f}"
)

# Date anomaly handling checks: flagged future dates must not enter monthly trends,
# while all original rows remain available for non-date KPIs.
if "order_date" in df.columns:
    flagged_future_count = int((pd.to_datetime(df["order_date"], errors="coerce") > AS_OF_DATE).sum())
    record_check(
        "Future-dated rows are excluded from date trends",
        int(trend_source.shape[0]) == int(df["order_date"].notna().sum() - flagged_future_count),
        f"future-dated rows flagged={flagged_future_count}; rows in trend source={len(trend_source)}"
    )
    record_check(
        "Source dataset row count is preserved",
        len(df) == len(analysis_df),
        f"source rows={len(df)}; analysis-copy rows={len(analysis_df)}"
    )

validation_results = pd.DataFrame(checks)
display(validation_results)
if not validation_results["Passed"].all():
    raise AssertionError("One or more KPI validation checks failed. Review validation_results.")

# Status-exclusion scenario: excludes Cancelled, Returned, and Return; retains other statuses,
# including rows with missing status. This is illustrative until the business rule is confirmed.
excluded_statuses = {"Cancelled", "Returned", "Return"}
status_text = df["delivery_status"].astype("string")
eligible_mask = ~status_text.isin(excluded_statuses)
scenario_revenue = float(revenue_values.loc[eligible_mask].sum())
scenario_orders = int(df.loc[eligible_mask, "order_id"].nunique())
scenario_aov = scenario_revenue / scenario_orders if scenario_orders else np.nan

revenue_sensitivity = pd.DataFrame([
    {"Scenario": "Baseline: all rows/statuses", "Revenue": total_revenue,
     "Distinct orders in included rows": total_orders,
     "AOV using included revenue / included orders": average_order_value},
    {"Scenario": "Exclude Cancelled, Returned, Return", "Revenue": scenario_revenue,
     "Distinct orders in included rows": scenario_orders,
     "AOV using included revenue / included orders": scenario_aov},
])
display(revenue_sensitivity.style.format({
    "Revenue": "{:,.2f}",
    "AOV using included revenue / included orders": "{:,.2f}"
}))

print(f"Rows with missing category (excluded from category ranking): {int(df['category'].isna().sum()):,}")
print(f"Rows with missing product name (product ID retained in product ranking): {int(df['product_name'].isna().sum()):,}")
print(f"Rows with missing delivery status: {int(df['delivery_status'].isna().sum()):,}")


,Check,Passed,Details
0,Revenue matches independent sum,True,calculated=2300662.71; independent=2300662.71
1,Order count matches distinct-ID count,True,calculated=5009; independent=5009
2,Units match independent sum,True,calculated=37508.00; independent=37508.00
3,AOV formula reconciles,True,calculated=459.31; independent=459.31
4,Revenue by status reconciles to total,True,dataset total=2300662.71; status sum=2300662.71
5,Order IDs are non-null,True,missing order IDs=0
6,Revenue values are non-negative,True,negative revenue rows=0
7,Category aggregation reconciles to rows with a...,True,category units total=37060.00
8,Product aggregation reconciles to total revenue,True,product revenue total=2300662.71; total revenu...
9,Future-dated rows are excluded from date trends,True,future-dated rows flagged=6; rows in trend sou...


,Scenario,Revenue,Distinct orders in included rows,AOV using included revenue / included orders
0,Baseline: all rows/statuses,"2,300,662.71",5009,459.31
1,"Exclude Cancelled, Returned, Return","2,064,988.19",4712,438.24


Rows with missing category (excluded from category ranking): 120
Rows with missing product name (product ID retained in product ranking): 79
Rows with missing delivery status: 179


## 11. Findings and decisions to carry into the report

- Baseline revenue is calculated across all rows and all delivery statuses because no approved revenue-recognition rule was supplied.
- Total orders counts distinct `order_id` values; repeated IDs are treated as order line items, not duplicate orders.
- Six rows dated **15 January 2035** are flagged as future-dated. Their dates are set to `NaT` only in `analysis_df`, so they do not distort monthly/category trend tables. Their values remain in non-date KPIs pending data-owner confirmation.
- The dataset also contains negative quantities and negative unit prices. These are surfaced by the quality checks and should be investigated before presenting unit totals or product-level interpretations as final.
- `Cancelled`, `Returned`, `Return`, and missing delivery statuses make revenue interpretation uncertain. Compare the baseline and status-exclusion scenario, then document the agreed business rule.
- No rows should be deleted or permanently date-corrected based only on this notebook. Keep the source file unchanged and document any approved correction in the data-quality log.